# Data Preparation

Download the CUB-200-2011 dataset, build a focused 20-species subset,
and run all preparation steps (inspection, label verification,
corrupted-image detection, and class-balance analysis) on that subset.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wenewone/cub2002011")

print("Path to dataset files:", path)

## 1. Dataset Setup

The downloaded dataset path is converted to a `Path` object and the main
CUB-200-2011 root folder is located.

In [ ]:
from pathlib import Path
import os
import shutil

# Convert the downloaded dataset path into a Path object
dataset_path = Path(path)

print("Dataset path:", dataset_path)
print("Dataset exists:", dataset_path.exists())

print("\nFiles and folders inside the dataset:")
for item in sorted(dataset_path.iterdir()):
    if item.is_dir():
        print("[Folder]", item.name)
    else:
        print("[File]", item.name)

In [ ]:
# Select the main CUB-200-2011 dataset folder
DATASET_ROOT   = dataset_path / "CUB_200_2011"
IMAGES_FOLDER  = DATASET_ROOT / "images"

print("Dataset root:", DATASET_ROOT)
print("Images folder exists:", IMAGES_FOLDER.exists())

print("\nFiles and folders inside CUB_200_2011:")
for item in sorted(DATASET_ROOT.iterdir()):
    print("-", item.name)

## 2. Build the 20-Species Subset

Before any analysis begins, the first 20 bird-species folders are copied
from the full dataset into a dedicated subset directory. Every subsequent
step in this notebook operates exclusively on this subset.

In [ ]:
# Destination directory for the 20-species subset
SUBSET_DIR            = "./dataset_20_species"
SUBSET_IMAGES_FOLDER  = Path(SUBSET_DIR)

# Remove any existing subset directory and start fresh
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
os.makedirs(SUBSET_DIR)

# Select the first 20 species folders (sorted alphabetically)
species_folders  = sorted([f for f in IMAGES_FOLDER.iterdir() if f.is_dir()])
selected_species = species_folders[:20]

# Copy each selected species folder into the subset directory
for species_path in selected_species:
    dst = os.path.join(SUBSET_DIR, species_path.name)
    shutil.copytree(str(species_path), dst)

print(f"20-species subset created at: {SUBSET_DIR}")
print(f"Number of species included:   {len(selected_species)}")
print("\nSpecies included:")
for species_path in selected_species:
    print(f"  - {species_path.name}")

## 3. Dataset Inspection

The 20-species subset folders and image counts are examined to confirm
the subset was assembled correctly.

In [ ]:
print("Subset path:", SUBSET_IMAGES_FOLDER)
print("Subset exists:", SUBSET_IMAGES_FOLDER.exists())

print("\nFolders inside the subset:")
for item in sorted(SUBSET_IMAGES_FOLDER.iterdir()):
    if item.is_dir():
        image_count = sum(
            1 for f in item.iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png")
        )
        print(f"  [Folder] {item.name}  ({image_count} images)")

## 4. Check Dataset Labels

The dataset label files are loaded from the full CUB-200-2011 metadata
and filtered to retain only the 20 selected species. The filtered records
are then checked for missing labels, duplicate IDs, and invalid class IDs.

In [ ]:
import pandas as pd

# File containing class IDs and bird species names
classes = pd.read_csv(
    DATASET_ROOT / "classes.txt",
    sep=r"\s+",
    names=["class_id", "class_name"]
)

# File containing image IDs and image paths
images = pd.read_csv(
    DATASET_ROOT / "images.txt",
    sep=r"\s+",
    names=["image_id", "image_path"]
)

# File connecting every image ID to a class ID
image_labels = pd.read_csv(
    DATASET_ROOT / "image_class_labels.txt",
    sep=r"\s+",
    names=["image_id", "class_id"]
)

# Names of the 20 selected species (folder name == class_name in this dataset)
selected_species_names = [p.name for p in selected_species]

# Keep only the classes that belong to the 20-species subset
classes_subset = classes[
    classes["class_name"].isin(selected_species_names)
].copy()

print("Total classes in full dataset:", len(classes))
print("Classes retained in subset:",    len(classes_subset))

In [ ]:
print("20-species subset classes:")
display(classes_subset)

In [ ]:
# Combine image paths, class IDs, and class names into one table
dataset_info = images.merge(
    image_labels,
    on="image_id",
    how="left"
)

dataset_info = dataset_info.merge(
    classes,
    on="class_id",
    how="left"
)

# Filter to the 20 selected species only
dataset_info = dataset_info[
    dataset_info["class_name"].isin(selected_species_names)
].copy()

dataset_info.reset_index(drop=True, inplace=True)

print("Subset dataset shape:", dataset_info.shape)
display(dataset_info.head(10))

In [ ]:
# Count missing values in each column
print("Missing values:")
print(dataset_info.isnull().sum())

# Check whether any image ID appears more than once
duplicate_image_ids = dataset_info["image_id"].duplicated().sum()

# Check whether any image path appears more than once
duplicate_image_paths = dataset_info["image_path"].duplicated().sum()

# Check whether class IDs are within the expected range for the subset
valid_class_ids = classes_subset["class_id"].tolist()
invalid_class_ids = dataset_info[
    ~dataset_info["class_id"].isin(valid_class_ids)
]

print("\nDuplicate image IDs:",  duplicate_image_ids)
print("Duplicate image paths:", duplicate_image_paths)
print("Invalid class IDs:",     len(invalid_class_ids))

In [ ]:
# Extract the folder name from each image path
# Example:
# 001.Black_footed_Albatross/image_001.jpg
# becomes:
# 001.Black_footed_Albatross

dataset_info["folder_name"] = dataset_info["image_path"].apply(
    lambda image_path: Path(image_path).parts[0]
)

# Find rows where the folder name and official class name do not match
label_mismatches = dataset_info[
    dataset_info["folder_name"] != dataset_info["class_name"]
]

print("Total images checked:",    len(dataset_info))
print("Folder-label mismatches:", len(label_mismatches))

if len(label_mismatches) == 0:
    print("Result: All image folders match their official class labels.")
else:
    print("Images with mismatched labels:")
    display(label_mismatches.head(10))

In [ ]:
# Build the full image path pointing to the SUBSET directory
dataset_info["full_image_path"] = dataset_info["image_path"].apply(
    lambda image_path: SUBSET_IMAGES_FOLDER / image_path
)

# Check whether every listed image exists in the subset folder
dataset_info["file_exists"] = dataset_info["full_image_path"].apply(
    lambda image_path: image_path.exists()
)

missing_image_files = dataset_info[
    dataset_info["file_exists"] == False
]

print("Images listed in metadata:", len(dataset_info))
print("Existing image files:",       dataset_info["file_exists"].sum())
print("Missing image files:",        len(missing_image_files))

if len(missing_image_files) == 0:
    print("Result: Every image listed in the metadata exists in the subset folder.")
else:
    display(missing_image_files.head(10))

### Label Verification Conclusion

The 20-species subset was verified against the CUB-200-2011 metadata files.
No missing labels, duplicate image IDs, duplicate image paths, or invalid
class IDs were found. Every image listed in the filtered metadata exists
in the subset folder.

## 5. Detect Corrupted and Duplicate Images

Each image in the 20-species subset is opened and verified to identify
corrupted or unreadable files. Exact duplicate images are detected using
their SHA-256 file hashes. The original subset is preserved, and invalid
or duplicate records are excluded from a clean dataset table.

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

def validate_image(image_path):
    """
    Check whether an image can be opened, verified and converted to RGB.
    """
    try:
        # Verify the image file structure
        with Image.open(image_path) as image:
            image.verify()

        # Open it again and check that pixel data can be loaded
        with Image.open(image_path) as image:
            image.convert("RGB").load()

        return True, ""

    except Exception as error:
        return False, str(error)


validation_results = []

for image_path in tqdm(
    dataset_info["full_image_path"],
    desc="Checking image files"
):
    is_valid, error_message = validate_image(image_path)

    validation_results.append({
        "is_valid_image":   is_valid,
        "validation_error": error_message
    })

validation_results = pd.DataFrame(validation_results)

dataset_info["is_valid_image"]   = validation_results["is_valid_image"]
dataset_info["validation_error"] = validation_results["validation_error"]

corrupted_images = dataset_info[
    dataset_info["is_valid_image"] == False
]

print("Total images checked:",             len(dataset_info))
print("Valid images:",                      dataset_info["is_valid_image"].sum())
print("Corrupted or unreadable images:",   len(corrupted_images))

if len(corrupted_images) == 0:
    print("Result: No corrupted images were found.")
else:
    display(
        corrupted_images[
            ["image_id", "image_path", "class_name", "validation_error"]
        ].head(10)
    )

In [ ]:
import hashlib

def calculate_file_hash(image_path, chunk_size=8192):
    """
    Calculate a SHA-256 hash from the image file contents.
    Identical files will produce the same hash.
    """
    sha256 = hashlib.sha256()

    with open(image_path, "rb") as image_file:
        while True:
            data = image_file.read(chunk_size)

            if not data:
                break

            sha256.update(data)

    return sha256.hexdigest()


image_hashes = []

for image_path, is_valid in tqdm(
    zip(
        dataset_info["full_image_path"],
        dataset_info["is_valid_image"]
    ),
    total=len(dataset_info),
    desc="Calculating image hashes"
):
    if is_valid:
        image_hashes.append(calculate_file_hash(image_path))
    else:
        image_hashes.append(None)

dataset_info["file_hash"] = image_hashes

# Keep the first copy and mark later identical files as duplicates
dataset_info["is_exact_duplicate"] = (
    dataset_info["file_hash"].notna()
    & dataset_info["file_hash"].duplicated(keep="first")
)

duplicate_images = dataset_info[
    dataset_info["is_exact_duplicate"] == True
]

print("Total valid images:",    dataset_info["is_valid_image"].sum())
print("Exact duplicate images:", len(duplicate_images))

if len(duplicate_images) == 0:
    print("Result: No exact duplicate images were found.")
else:
    print("Duplicate images that will be excluded:")
    display(
        duplicate_images[
            ["image_id", "image_path", "class_name", "file_hash"]
        ].head(10)
    )

In [ ]:
clean_dataset_info = dataset_info[
    (dataset_info["is_valid_image"]    == True)
    & (dataset_info["is_exact_duplicate"] == False)
].copy()

clean_dataset_info.reset_index(drop=True, inplace=True)

removed_corrupted = len(
    dataset_info[dataset_info["is_valid_image"] == False]
)

removed_duplicates = len(
    dataset_info[dataset_info["is_exact_duplicate"] == True]
)

print("Original number of images:", len(dataset_info))
print("Corrupted images excluded:", removed_corrupted)
print("Duplicate images excluded:", removed_duplicates)
print("Clean dataset size:",        len(clean_dataset_info))

In [ ]:
# Get the hash values that appear more than once
duplicate_hashes = dataset_info.loc[
    dataset_info["file_hash"].duplicated(keep=False),
    "file_hash"
].unique()

# Display every image sharing a duplicate hash
duplicate_groups = dataset_info[
    dataset_info["file_hash"].isin(duplicate_hashes)
][
    ["image_id", "image_path", "class_id", "class_name", "file_hash"]
].sort_values("file_hash")

print("Number of duplicate groups:",      len(duplicate_hashes))
print("Images involved in duplicate groups:", len(duplicate_groups))

display(duplicate_groups)

In [ ]:
duplicate_class_check = (
    duplicate_groups
    .groupby("file_hash")["class_id"]
    .nunique()
)

cross_class_duplicates = duplicate_class_check[
    duplicate_class_check > 1
]

print(
    "Duplicate groups containing images from different classes:",
    len(cross_class_duplicates)
)

if len(cross_class_duplicates) == 0:
    print("Result: The duplicate images belong to the same bird class.")
else:
    print("Warning: An identical image appears under different class labels.")

### Image Quality and Duplicate Check Conclusion

All images in the 20-species subset were opened and verified successfully.
SHA-256 file hashes were computed to detect exact duplicates. Any corrupted
or duplicate images are excluded from the clean dataset table. The original
subset folder is left untouched.

## 6. Class Distribution and Balance

The number of clean images in each of the 20 bird-species classes is
examined to determine whether class balancing is necessary. Balancing
will only be applied if there is a meaningful difference between class sizes.

In [ ]:
# Count valid and unique images in each bird class
class_distribution = (
    clean_dataset_info
    .groupby(["class_id", "class_name"])
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count")
)

minimum_count  = class_distribution["image_count"].min()
maximum_count  = class_distribution["image_count"].max()
average_count  = class_distribution["image_count"].mean()
imbalance_ratio = maximum_count / minimum_count

print("Number of classes:",            len(class_distribution))
print("Minimum images in one class:",  minimum_count)
print("Maximum images in one class:",  maximum_count)
print("Average images per class:",     round(average_count, 2))
print("Imbalance ratio:",              round(imbalance_ratio, 3))

print("\nClasses with the fewest images:")
display(class_distribution.head(10))

print("\nClasses with the most images:")
display(class_distribution.tail(10))

In [ ]:
# Show how many classes have each possible number of images
count_frequency = (
    class_distribution["image_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

count_frequency.columns = [
    "images_per_class",
    "number_of_classes"
]

display(count_frequency)

In [ ]:
import matplotlib.pyplot as plt

plot_distribution = class_distribution.sort_values("class_id")

plt.figure(figsize=(12, 5))

plt.bar(
    plot_distribution["class_id"],
    plot_distribution["image_count"]
)

plt.axhline(
    average_count,
    linestyle="--",
    label=f"Average = {average_count:.2f}"
)

plt.xlabel("Bird Class ID")
plt.ylabel("Number of Clean Images")
plt.title("Class Distribution of the 20-Species Subset")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# A ratio close to 1 means that the classes have similar sizes.
# We avoid balancing when the difference is small because undersampling
# would unnecessarily discard useful images.

if imbalance_ratio <= 1.5:
    balancing_required = False

    print("Balancing required: No")
    print(
        "Reason: The class sizes are sufficiently similar, "
        "so balancing would unnecessarily remove useful images."
    )
    print(
        "Decision: Keep all clean images and use a stratified "
        "training/testing split later."
    )
else:
    balancing_required = True

    print("Balancing required: Yes")
    print(
        "Reason: There is a considerable difference between "
        "the smallest and largest classes."
    )

In [ ]:
# No resampling is performed when the dataset is sufficiently balanced.
# The cleaned dataset becomes the final prepared dataset.

if not balancing_required:
    final_dataset_info = clean_dataset_info.copy()
else:
    # Keep the clean dataset temporarily.
    # A balancing method should only be chosen after team agreement.
    final_dataset_info = clean_dataset_info.copy()

final_dataset_info.reset_index(drop=True, inplace=True)

print("Clean dataset size:",         len(clean_dataset_info))
print("Final prepared dataset size:", len(final_dataset_info))
print("Number of final classes:",     final_dataset_info["class_id"].nunique())

### Class-Balance Conclusion

The 20-species subset class-distribution analysis showed whether the class
sizes are sufficiently similar. If the imbalance ratio is at or below 1.5,
no undersampling or oversampling is applied — all clean images are retained
and a stratified training/testing split will be used later to preserve
class representation.

In [ ]:
# Keep only portable columns.
# Do not save full_image_path because it contains a local machine path.

columns_to_save = [
    "image_id",
    "image_path",
    "class_id",
    "class_name"
]

prepared_metadata = final_dataset_info[
    columns_to_save
].copy()

output_file = "/content/clean_dataset_metadata.csv"
prepared_metadata.to_csv(output_file, index=False)

print("Metadata saved to:", output_file)
print("Saved rows:",        len(prepared_metadata))

display(prepared_metadata.head())